# Notebook 10 — Amélioration du modèle

## Contexte
Le meilleur modèle actuel est `05_tabular` (skrub + catégorielles) : **RMSE=7.53, MAE=5.93, R²=0.79**.
Le feature engineering du notebook 07 n'a pas amélioré les résultats.

## Stratégie d'amélioration
1. Tuner les hyperparamètres HGBR dans le pipeline skrub
2. Tester sans `rater_id` (risque de biais évaluateur)
3. Augmenter `max_iter` (plus d'arbres)
4. Réduire `learning_rate` + augmenter `max_iter`
5. Contraindre la profondeur (`max_depth`)

## Fichiers utilisés
- `data/X_train.csv`, `data/y_train.csv`, `data/X_test.csv`

## Notebook suivant
`11_lgbm_advanced.ipynb` — LightGBM et features avancées de trajectoire patient.

## 1. Imports

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
import skore
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from skrub import TableVectorizer, tabular_pipeline

## 2. Connexion Skore Hub

In [2]:
config = json.loads(Path("../.skore").read_text())
os.environ["SKORE_HUB_API_KEY"] = config["api_key"]
skore.login()
hub_project = skore.Project(
    "bobathon-esilv",
    mode="hub",
    workspace=config["workspace"],
)
print(f"Connecté : workspace={hub_project.workspace}")

╭───────────────────────────────── Login to Skore Hub ─────────────────────────────────╮
│                                                                                      │
│                        Successfully logged in, using API key.                        │
│                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────╯

Connecté : workspace=BERNIE MADOFF


## 3. Chargement des données

In [3]:
DATA_DIR = Path("../data")

X_train_raw = pd.read_csv(DATA_DIR / "X_train.csv")
y_train_raw = pd.read_csv(DATA_DIR / "y_train.csv")
X_test_raw  = pd.read_csv(DATA_DIR / "X_test.csv")

visits = X_train_raw.merge(y_train_raw, on="Index", how="inner", validate="one_to_one")
assert len(visits) == len(X_train_raw)
assert visits["target"].notna().all()

# Features : tout sauf Index, patient_id, target
X_full = visits.drop(columns=["Index", "patient_id", "target"])
y      = visits["target"]
groups = visits["patient_id"]

# Features sans rater_id
X_no_rater = X_full.drop(columns=["rater_id"])

cv = list(GroupKFold(n_splits=5).split(X_full, y, groups=groups))
cv_nr = list(GroupKFold(n_splits=5).split(X_no_rater, y, groups=groups))

print(f"X_full     : {X_full.shape}")
print(f"X_no_rater : {X_no_rater.shape}")
print(f"Colonnes   : {list(X_full.columns)}")

X_full     : (44590, 11)
X_no_rater : (44590, 10)
Colonnes   : ['cohort', 'sexM', 'gene', 'age_at_diagnosis', 'age', 'ledd', 'time_since_intake_on', 'time_since_intake_off', 'rater_id', 'on', 'off']


## 4. Référence — tabular_pipeline par défaut (RMSE cible : 7.53)

In [4]:
ref_model = tabular_pipeline("regressor")
report_ref = skore.evaluate(ref_model, X_full, y, splitter=cv)
rmse_ref = float(report_ref.metrics.rmse().iloc[:, 0].values.mean())
mae_ref  = float(report_ref.metrics.mae().iloc[:, 0].values.mean())
print(f"Référence  — RMSE={rmse_ref:.4f}  MAE={mae_ref:.4f}")

Output()

Référence  — RMSE=7.5172  MAE=5.9154


## 5. Grille d'hyperparamètres HGBR

On construit manuellement le pipeline `TableVectorizer + HGBR` pour pouvoir passer des hyperparamètres au HGBR.

In [5]:
from sklearn.preprocessing import OrdinalEncoder
from skrub import ToCategorical

def make_hgbr_pipeline(max_iter=100, learning_rate=0.1, max_depth=None,
                        min_samples_leaf=20, l2_reg=0.0):
    return Pipeline([
        ("vectorizer", TableVectorizer(low_cardinality=ToCategorical())),
        ("hgbr", HistGradientBoostingRegressor(
            max_iter=max_iter,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            l2_regularization=l2_reg,
            random_state=0,
        )),
    ])

configs = [
    # (label, max_iter, lr, max_depth, min_samples_leaf, l2_reg)
    ("default",             100,  0.10, None,  20, 0.0),
    ("more_trees",          300,  0.10, None,  20, 0.0),
    ("slow_lr",             500,  0.05, None,  20, 0.0),
    ("depth_5",             300,  0.10,    5,  20, 0.0),
    ("depth_7",             300,  0.10,    7,  20, 0.0),
    ("min_leaf_10",         300,  0.10, None,  10, 0.0),
    ("min_leaf_5",          300,  0.10, None,   5, 0.0),
    ("l2_reg",              300,  0.10, None,  20, 1.0),
    ("slow_deep",           500,  0.05,    7,  10, 0.0),
]

results = []
for label, n_iter, lr, depth, leaf, l2 in configs:
    model = make_hgbr_pipeline(n_iter, lr, depth, leaf, l2)
    rep = skore.evaluate(model, X_full, y, splitter=cv)
    rmse = float(rep.metrics.rmse().iloc[:, 0].values.mean())
    mae  = float(rep.metrics.mae().iloc[:, 0].values.mean())
    results.append({"config": label, "max_iter": n_iter, "lr": lr,
                    "max_depth": depth, "min_leaf": leaf, "l2": l2,
                    "RMSE": rmse, "MAE": mae})
    print(f"{label:20s}  RMSE={rmse:.4f}  MAE={mae:.4f}")

results_df = pd.DataFrame(results).sort_values("RMSE")
print()
display(results_df.reset_index(drop=True).round(4))

Output()

Output()

default               RMSE=7.5232  MAE=5.9290


Output()

more_trees            RMSE=7.5601  MAE=5.9486


Output()

... (8 outputs left out)

Output()

l2_reg                RMSE=7.5389  MAE=5.9329


slow_deep             RMSE=7.5198  MAE=5.9186



,config,max_iter,lr,max_depth,min_leaf,l2,RMSE,MAE
0,slow_lr,500,0.05,NaN,20,0.0,7.5112,5.9087
1,slow_deep,500,0.05,7.0,10,0.0,7.5198,5.9186
2,depth_5,300,0.10,5.0,20,0.0,7.5232,5.9170
3,default,100,0.10,NaN,20,0.0,7.5232,5.9290
4,l2_reg,300,0.10,NaN,20,1.0,7.5389,5.9329
5,depth_7,300,0.10,7.0,20,0.0,7.5409,5.9318
6,min_leaf_5,300,0.10,NaN,5,0.0,7.5486,5.9401
7,min_leaf_10,300,0.10,NaN,10,0.0,7.5487,5.9377
8,more_trees,300,0.10,NaN,20,0.0,7.5601,5.9486


## 6. Impact de rater_id — meilleure config sans rater_id

In [6]:
# Prendre la meilleure config trouvée ci-dessus
best_row = results_df.iloc[0]
best_label = best_row["config"]
print(f"Meilleure config : {best_label}")
print(f"  RMSE avec rater_id    : {best_row['RMSE']:.4f}")

best_model_no_rater = make_hgbr_pipeline(
    int(best_row["max_iter"]),
    float(best_row["lr"]),
    None if pd.isna(best_row["max_depth"]) else int(best_row["max_depth"]),
    int(best_row["min_leaf"]),
    float(best_row["l2"]),
)
rep_nr = skore.evaluate(best_model_no_rater, X_no_rater, y, splitter=cv_nr)
rmse_nr = float(rep_nr.metrics.rmse().iloc[:, 0].values.mean())
mae_nr  = float(rep_nr.metrics.mae().iloc[:, 0].values.mean())
print(f"  RMSE sans rater_id    : {rmse_nr:.4f}  MAE={mae_nr:.4f}")

Output()

Meilleure config : slow_lr
  RMSE avec rater_id    : 7.5112


  RMSE sans rater_id    : 7.7690  MAE=6.1123


## 7. Sélection et envoi du meilleur modèle sur Skore Hub

In [7]:
# Choisir : avec ou sans rater_id selon les résultats ci-dessus
# Par défaut on garde la version avec rater_id si elle est meilleure
best_rmse_with = float(results_df.iloc[0]["RMSE"])
use_rater_id = best_rmse_with <= rmse_nr

if use_rater_id:
    final_X_cv, final_cv, final_label = X_full, cv, "avec rater_id"
    best_report = skore.evaluate(
        make_hgbr_pipeline(
            int(results_df.iloc[0]["max_iter"]),
            float(results_df.iloc[0]["lr"]),
            None if pd.isna(results_df.iloc[0]["max_depth"]) else int(results_df.iloc[0]["max_depth"]),
            int(results_df.iloc[0]["min_leaf"]),
            float(results_df.iloc[0]["l2"]),
        ),
        X_full, y, splitter=cv
    )
else:
    final_X_cv, final_cv, final_label = X_no_rater, cv_nr, "sans rater_id"
    best_report = rep_nr

hub_project.put("08_improved", best_report)
print(f"Modèle sélectionné : {final_label}")
print(f"Rapport envoyé : 08_improved")

Output()

Output()

Consult your report at https://skore.probabl.ai/BERNIE 
MADOFF/bobathon-esilv/cross-validations/61484

Modèle sélectionné : avec rater_id
Rapport envoyé : 08_improved


## 8. Entraînement final sur toutes les données et soumission

In [8]:
best_row = results_df.iloc[0]

final_model = make_hgbr_pipeline(
    int(best_row["max_iter"]),
    float(best_row["lr"]),
    None if pd.isna(best_row["max_depth"]) else int(best_row["max_depth"]),
    int(best_row["min_leaf"]),
    float(best_row["l2"]),
)

# Préparer X_test avec les mêmes colonnes que le train
X_test_model = X_test_raw.drop(columns=["Index", "patient_id"])
if not use_rater_id:
    X_test_model = X_test_model.drop(columns=["rater_id"])

final_model.fit(final_X_cv if use_rater_id else X_no_rater, y)
predictions = final_model.predict(X_test_model)

submission = X_test_raw[["Index"]].copy()
submission["target"] = predictions

assert list(submission.columns) == ["Index", "target"]
assert len(submission) == len(X_test_raw)
assert submission["Index"].equals(X_test_raw["Index"])
assert submission["target"].notna().all()

Path("../submissions").mkdir(exist_ok=True)
submission.to_csv("../submissions/improved.csv", index=False)

print("Vérifications : OK")
print(f"Fichier : submissions/improved.csv ({len(submission)} lignes)")
display(submission.head())

Vérifications : OK
Fichier : submissions/improved.csv (11013 lignes)


,Index,target
0,66,20.078095
1,67,29.189462
2,68,29.703032
3,69,34.024595
4,70,32.816645


## 9. Conclusion

**Récap des RMSE (GroupKFold) :**

| Modèle | RMSE | MAE |
|---|---|---|
| Baseline (dummy) | 16.48 | 13.48 |
| Ridge | 10.49 | 8.44 |
| HGBR numérique | 7.75 | 6.10 |
| tabular_pipeline (défaut) | 7.53 | 5.93 |
| **Modèle amélioré** | **voir ci-dessus** | **voir ci-dessus** |

**Fichier soumission** : `submissions/improved.csv`  
**Rapport Hub** : `08_improved`